# 第1回　ガイダンスと「直感」の敗北
## ―― なぜ統計が必要か

統計学Ⅰ（B）　／　北星学園大学

---

### このノートの使い方

**今日はプログラミングはしない。** 各セルの左にある ▶ ボタンを上から順に押して、結果を**自分の目で見る**だけでよい。コードの中身は分からなくて構わない（第4回でちゃんとやる）。

大事なのはただ一つ ――

> **あなたの「直感」と、データが出す「答え」を、見比べること。**

In [ ]:
# まず準備（グラフを日本語表示できるようにする）。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 直感クイズ ①　モンティ・ホール問題

3つのドアがある。1つが当たり（新車）、2つがハズレ（ヤギ）。

1. あなたはドアを1つ選ぶ。
2. 司会者が、**残り2つのうちハズレのドアを1つ**開けて見せる。
3. 司会者「ドアを**変えてもいい**ですよ」

**問い：あなたは選び直すべきか？　それとも最初の選択を貫くべきか？**

多くの人は「2択になったんだから50:50。変えても同じ」と感じる。

まず**自分の予想を決めてから**、次のセルを実行しよう。

In [ ]:
# モンティ・ホールを 1万回くりかえして、勝率を数える
rng = np.random.default_rng(0)
N = 10000

当たりのドア = rng.integers(0, 3, N)   # 当たりがどこにあるか
最初の選択   = rng.integers(0, 3, N)   # あなたが最初に選ぶドア

# 「変えない」で勝つ = 最初の選択がそのまま当たりだったとき
変えない勝率 = (最初の選択 == 当たりのドア).mean()
# 「変える」で勝つ = 最初の選択がハズレだったとき（変えれば必ず当たりに行ける）
変える勝率   = (最初の選択 != 当たりのドア).mean()

print(f"変えない場合の勝率： {変えない勝率:.1%}")
print(f"変える場合の勝率　： {変える勝率:.1%}")

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(["変えない", "変える"], [変えない勝率, 変える勝率], color=["#888", "#e8503a"])
plt.ylabel("勝率")
plt.title("モンティ・ホール：1万回シミュレーション")
plt.ylim(0, 1)
for i, v in enumerate([変えない勝率, 変える勝率]):
    plt.text(i, v + 0.02, f"{v:.1%}", ha="center")
plt.show()

**直感は「50:50」と言った。データは「変えれば約2/3」と言う。**

直感は *なんとなく* 間違えたのではない。**いつも同じ向きに**（変えない方に賭けてしまう向きに）間違える。これを系統的バイアスと呼ぶ。

---
## 直感クイズ ②　誕生日のパラドックス

この教室に **23人** いるとする。

**問い：その中に「誕生日が同じ2人」がいる確率は、だいたい何%だと思う？**

365日もあるんだから、23人くらいじゃ、ほとんど被らない気がする……？

予想を決めてから実行しよう。

In [ ]:
def 同じ誕生日が出る確率(人数, 試行=20000):
    r = np.random.default_rng(1)
    かぶった = 0
    for _ in range(試行):
        誕生日 = r.integers(0, 365, 人数)
        if len(set(誕生日)) < 人数:   # 重複があれば
            かぶった += 1
    return かぶった / 試行

print(f"23人で誕生日が同じペアがいる確率： {同じ誕生日が出る確率(23):.1%}")

In [ ]:
# 人数を増やすと確率がどう上がるか、曲線で見る
人数リスト = list(range(2, 51))
確率リスト = [同じ誕生日が出る確率(k, 試行=4000) for k in 人数リスト]

plt.figure(figsize=(7, 4))
plt.plot(人数リスト, 確率リスト, marker="o", ms=3, color="#e8503a")
plt.axhline(0.5, ls="--", color="gray")
plt.axvline(23, ls="--", color="gray")
plt.xlabel("人数")
plt.ylabel("誕生日が同じペアがいる確率")
plt.title("誕生日のパラドックス")
plt.show()

**たった23人で50%を超える。** 直感は「ほとんど被らない」と言ったはずだ。

なぜ外れるのか？　直感は「**自分**と同じ誕生日の人」を探してしまう。本当の問いは「**誰かと誰か**が同じ」。ペアの数は人数とともに爆発的に増える（23人なら253ペア）。直感は組み合わせの爆発を過小評価する。

---
## 直感クイズ ③　少ない試行は、当てにならない

コインを投げる。表が出る確率はもちろん 1/2。

**問い：10回投げたら、ちょうど表が5回ずつ出る……とは限らない。では何回も投げ続けると、表の割合はどうなる？**

In [ ]:
r = np.random.default_rng(7)
コイン = r.integers(0, 2, 1000)            # 0=裏, 1=表 を1000回
表の割合 = np.cumsum(コイン) / np.arange(1, 1001)  # 1投目までの割合, 2投目まで, ...

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, 1001), 表の割合, color="#e8503a")
plt.axhline(0.5, ls="--", color="gray")
plt.xlabel("投げた回数")
plt.ylabel("表が出た割合")
plt.title("投げるほど 0.5 に近づく（大数の法則）")
plt.ylim(0, 1)
plt.show()

最初の数回はガタガタに偏る。「3連続で表が出た、次は裏が来るはず」――これも直感の罠（ギャンブラーの誤謬）。

**少ないデータで結論を出すと、偶然の偏りを『意味』だと勘違いする。** だから統計は「どれくらいのデータがあれば信じてよいか」を問題にする。

---
## 今日のまとめ

| 直感が言ったこと | データが示したこと |
|---|---|
| ① 変えても50:50 | 変えれば約 **2/3** |
| ② 23人じゃ被らない | **50%超** で被る |
| ③ 少ない回数でも傾向は分かる | 少数は **偶然に振り回される** |

直感は「速い」。だが**系統的に・同じ向きに**間違える。統計学は、その間違え方を**見える化して補正する**ための共通言語だ。

次回からは、この「補正の道具」を一つずつ手に入れていく。そして第4回からは、君自身がこのコードを書く側になる。

> **課題（Moodle）**：今日の直感クイズの答え合わせと、「自分の直感が外れた経験」のふりかえりを提出する。